# TP5 - Gallardo

In [ ]:
# importamos paquetes
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from IPython.display import display
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, AdaBoostRegressor, AdaBoostClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, roc_auc_score, mean_squared_error, roc_curve, RocCurveDisplay

In [ ]:
# ---------------------------------------------------------------
# Rutas del proyecto. El notebook vive en programs/ y lee de
# ../input/ y escribe en ../output/. Funciona tanto si se ejecuta
# desde programs/ como desde la raiz del TP.
# ---------------------------------------------------------------
from pathlib import Path

BASE = Path.cwd() if (Path.cwd() / "input").is_dir() else Path.cwd().parent
INPUT = BASE / "input"
OUTPUT = BASE / "output"
OUTPUT.mkdir(exist_ok=True)

print("Input :", INPUT)
print("Output:", OUTPUT)

In [ ]:
# importamos la base de datos limpia generada en el TP3
respondieron = pd.read_csv(INPUT / "respondieron.csv", low_memory=False)

print(respondieron.shape)
respondieron.head()

## Ejercicio 1: función evalua_metodo

In [ ]:
# helper: redondea solo si el valor es numerico (las metricas pueden venir como '.')
def _r(valor, dec=2):
    return round(valor, dec) if isinstance(valor, (int, float, np.integer, np.floating)) else valor

def evalua_metodo(modelo, X_train, y_train, X_test, y_test):
    '''
    Entrena y evalúa un modelo de machine learning en un conjunto de testeo, devolviendo métricas de desempeño.
    
    Input
    modelo : un estimador de scikit-learn
        Tiene que ser un modelo ya configurado, como RandomForestClassifier()
    
    X_train y X_test : numpy.ndarray de forma (n_muestras_entrenamiento, n_features)
        Matriz de características para entrenamiento y testeo

    y_train, y_test : numpy.ndarray de forma (n_muestras_entrenamiento,)
        Vector binario de etiquetas para entrenamiento y testeo 

    Output
    resultados, un diccionario con las siguientes claves:
        - 'Modelo' : nombre del modelo evaluado.
        - 'Híperparametros' : diccionario con los hiperparámetros del modelo.
        - 'Matriz de Confusión' : (solo si es clasificación) 
        - 'Accuracy' : exactitud (solo si es clasificación)
        - 'Precision' : precisión (solo si es clasificación)
        - 'Recall' : sensibilidad (solo si es clasificación)
        - 'MSE' : error cuadrático medio (siempre calculado)
        - 'AUC' : área bajo la curva ROC (si el modelo devuelve probabilidades)
        - 'FPR' : false positive rate (si se calcula ROC)
        - 'TPR' : true positive rate (si se calcula ROC)
    '''
    
    probabilidad = True
    estimador = type(modelo).__name__

    model = modelo.fit(X_train, y_train)    
    y_pred = model.predict(X_test)

    try:
        y_prob = model.predict_proba(X_test)[:,1]
    except:
        probabilidad = False

    mse = mean_squared_error(y_test, y_pred)

    try:
        matriz_confusion = confusion_matrix(y_test, y_pred)
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
    except: 
        matriz_confusion = accuracy = precision = recall = '.'
    
    if probabilidad:
        auc = roc_auc_score(y_test, y_prob)
        fpr, tpr, thresholds = roc_curve(y_test, y_prob, drop_intermediate=False)
    else:
        auc = fpr = tpr = thresholds = '.'
    
    resultados = {'Modelo': estimador, 'Híperparametros': modelo.get_params(), 'Matriz': matriz_confusion, 'Acc.': _r(accuracy), 'Prec.': _r(precision),
                  'Rec.': _r(recall), 'MSE': _r(mse), 'AUC': _r(auc), 'FPR': fpr, 'TPR': tpr}

    return resultados

## Ejercicio 2: función cross_validation

In [ ]:
def cross_validation(modelo, k, X, y):
    '''
    Realiza validación cruzada k-fold para un modelo de clasificación binaria y recopila métricas de desempeño. 
    Sirve para para entrenar y evaluar el modelo en `k` particiones distintas. En cada iteración, llama la función
    evalua_metodo para obtener métricas de desempeño.
     
    Input
    modelo : un estimador de scikit-learn
        Tiene que ser un modelo ya configurado, como RandomForestClassifier()
    
    k : int, número de particiones (folds) para la validación cruzada.
    
    X : numpy.ndarray de forma (n_muestras, n_features)
        Matriz de características
    
    y : numpy.ndarray de forma (n_muestras,)
        Vector binario de etiquetas objetivo

    Output
    metrics : diccionario con las listas de métricas por fold
        Las metricas provienen de la función evalua_metodo
    '''
    
    cv = KFold(n_splits=k, shuffle=True, random_state=444)
    metrics = {'Modelo': [], 'Híperparametros': [], 'Matriz': [], 'Acc.': [], 'Prec.': [],
                  'Rec.': [], 'MSE': [], 'AUC': [], 'FPR': [], 'TPR': []}

    for i, (train_index, valid_index) in enumerate(cv.split(X)):
        X_train, X_test = X[train_index], X[valid_index]
        y_train, y_test = y[train_index], y[valid_index]

        metric = evalua_metodo(modelo, X_train, y_train, X_test, y_test)
        
        for key in metrics.keys():
            metrics[key].append(metric[key])
    
    return metrics

## Ejercicio 3: función evalua_config

In [ ]:
def evalua_config(modelo, k, X, y, parametros):
    '''
    Selecciona la mejor configuración de hiperparámetros para un modelo mediante validación cruzada k-fold. 
    Evalúa todas las combinaciones posibles de hiperparámetros usando validación cruzada, seleccionando la configuración
    que minimiza el error cuadrático medio (MSE) promedio. Si el modelo admite el parámetro `max_iter`, se fija automáticamente
    en 100000 para evitar problemas de convergencia.

    Input
    modelo : un estimador de scikit-learn
        Debe ser la clase del modelo a evaluar, como 'RandomForestClassifier'
    
    k : int, número de particiones (folds) para la validación cruzada.
    
    X : numpy.ndarray de forma (n_muestras, n_features)
        Matriz de características
    
    y : numpy.ndarray de forma (n_muestras,)
        Vector binario de etiquetas objetivo

    parametros: diccionario con la listas de valores de hiperparámetros a evaluar.

    Output
    best_model : estimador de scikit-learn
        Instancia del modelo con la mejor combinación de hiperparámetros
    '''
    
    menor_error = np.inf
    best_model = None

    keys = list(parametros.keys())
    values = list(parametros.values())

    try:
        max_iter = 'max_iter' in modelo().get_params()
    except:
        max_iter = False
   
    for config in product(*values):
        parameter = dict(zip(keys, config))
        
        if max_iter:
            parameter['max_iter'] = 100000
        
        model = modelo(**parameter)
        metrics = cross_validation(model, k, X, y)
        error = np.mean(metrics['MSE'])

        if error < menor_error:
            menor_error = error
            best_model = model
    
    return best_model

## Ejercicio 4: función evalua_multiples_metodos

In [ ]:
def evalua_multiples_metodos(X, y, k=5, sufijo=''):
    '''
    Evalúa múltiples modelos de clasificación (o regresión) con selección de hiperparámetros y métricas comparativas.
    Esta función entrena, selecciona y compara distintos algoritmos (Logistic Regression, LDA, KNN, LASSO, Random Forest,
    AdaBoost y Gradient Boosting), aplicando búsqueda de hiperparámetros con validación cruzada k-fold. Los modelos se ajustan
    sobre un conjunto de entrenamiento (70%) y se evalúan sobre el conjunto de prueba (30%).

    Para cada modelo, se calcula una serie de métricas de desempeño y se genera la curva ROC correspondiente.
    La función devuelve una tabla resumen comparando el rendimiento de cada modelo (eligiendo el mejor modelo boosting).

    Input
    X : numpy.ndarray de forma (n_muestras, n_features)
        Matriz de características
    
    y : numpy.ndarray de forma (n_muestras,)
        Vector binario de etiquetas objetivo   
        
    k : int, opcional (default=5)
        Número de particiones (folds) para la validación cruzada.

    sufijo : str, opcional (default='')
        Cadena que se agregará al nombre de los archivos `.png` de las curvas ROC generadas, 
        útil para distinguir entre diferentes llamadas de la función y evitar sobrescritura de archivos.

    Output
    tabla : data frame de pandas
        Tabla comparativa con los resultados de cada modelo entrenado, contiene las metricas obtenidas con la función evalua_metodo
    
    curvas ROC: se muestran y se descargan como png
        Si se usa un sufijo, se incluye al final del nombre
        
    Nota: el modelo de boosting con peor desempeño (según mayor MSE) se excluye de la tabla final.
    '''

    modelos = []
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=444)
    
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)

    parameters_default = []

    # LOGIT
    logit = {'penalty': [None], 'solver': ['lbfgs'], 'fit_intercept': [True, False]}
    parameters_default.append(LogisticRegression().get_params())
    modelos.append(evalua_config(LogisticRegression, k, X_train, y_train, logit))
    
    # LDA
    parameters_default.append(LinearDiscriminantAnalysis().get_params())
    modelos.append(LinearDiscriminantAnalysis())
    
    # KNN
    knn = {'weights': ['uniform', 'distance'], 'n_neighbors': [3, 5, 7]}
    parameters_default.append(KNeighborsClassifier().get_params())
    modelos.append(evalua_config(KNeighborsClassifier, k, X_train, y_train, knn))

    # Lasso
    lasso = {'C': [0.1, 1, 10], 'solver': ['liblinear'], 'penalty': ['l1'], 'fit_intercept': [True, False]}
    parameters_default.append(LogisticRegression().get_params())
    modelos.append(evalua_config(LogisticRegression, k, X_train, y_train, lasso))

    # Random Forest, AdaBoosting, Gradient Boosting
    random_forest = {'random_state': [444], 'bootstrap': [True], 'max_samples': [0.25, 0.5, 0.75], 
                     'max_features': [3, 6, 9, "sqrt", None], 'n_estimators': [25, 50, 75, 100]}

    ada = {'random_state': [444], 'n_estimators': [25, 50, 75, 100], 'learning_rate': [0.001, 0.01, 0.1, 1, 1.5, 5]}

    gradient = {'random_state': [444], 'n_estimators': [25, 50, 75, 100], 
                'learning_rate': [0.001, 0.01, 0.1, 0.5, 0.75], 'max_depth': [None, 1, 2, 3, 4, 5]}
    
    if len(np.unique(y)) <= 10:
        rf = RandomForestClassifier
        ab = AdaBoostClassifier
        gb = GradientBoostingClassifier
    else:
        rf = RandomForestRegressor
        ab = AdaBoostRegressor
        gb = GradientBoostingRegressor

    modelos.append(evalua_config(rf, k, X_train, y_train, random_forest))
    modelos.append(evalua_config(ab, k, X_train, y_train, ada))
    modelos.append(evalua_config(gb, k, X_train, y_train, gradient))

    parameters_default.append(rf().get_params())
    parameters_default.append(ab().get_params())
    parameters_default.append(gb().get_params())

    # Resultados
    resultados = {'Modelo': [], 'Híperparametros': [], 'Matriz': [], 'Acc.': [], 'Prec.': [],
                  'Rec.': [], 'MSE': [], 'AUC': [], 'FPR': [], 'TPR': []}

    logit_check = True # para distinguir entre Logit y LASSO
    
    for modelo in modelos:
        resultado = evalua_metodo(modelo, X_train, y_train, X_test, y_test)

        if resultado['Modelo'] == 'LogisticRegression' and logit_check:
            logit_check = False
        elif resultado['Modelo'] == 'LogisticRegression':
            resultado['Modelo'] = 'LogisticRegression(LASSO)'
        
        nombre = resultado['Modelo']
        roc_display = RocCurveDisplay(fpr=resultado['FPR'], tpr=resultado['TPR'], roc_auc=resultado['AUC'], estimator_name=nombre)
        roc_display.plot()
        plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
        plt.title(f'Curva {nombre}')
        plt.savefig(OUTPUT / f'{nombre}{sufijo}.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        
        for key in resultados.keys():
            resultados[key].append(resultado[key])

    boosting = resultados['Modelo'][-1] if resultados['MSE'][-1] >= resultados['MSE'][-2] else resultados['Modelo'][-2]
    
    parameters = resultados['Híperparametros']
    hiperparametros = []
    
    for actual, default in zip(parameters, parameters_default):
        line = []
        for key, value in actual.items():
            if value != default[key] and key != 'max_iter' and key != 'random_state':
                line.append(f'{key} = {value}')
        hiperparametros.append(', '.join(line) if line else '')

    resultados['Híperparametros'] = hiperparametros
    
    tabla = pd.DataFrame(resultados)

    tabla = tabla[tabla['Modelo'] != boosting]
    tabla = tabla.drop(columns=['FPR', 'TPR'])
    
    return tabla

## Testeamos

In [ ]:
# separamos en X e y
variables = ['CONSTANTE', 'CH04', 'CH06', 'CH07', 'CH16', 'PERSONAS_HOGAR', 'EDAD2', 'OBRA_SOCIAL', 'EDUCACION', 'EDADxEDUCACION', 'COLEGIO_PRIVADO']

X_04 = respondieron.loc[respondieron['ANO4'] == 2004, variables].to_numpy()
y_04 = respondieron.loc[respondieron['ANO4'] == 2004, 'DESOCUPADOS'].to_numpy()

X_24 = respondieron.loc[respondieron['ANO4'] == 2024, variables].to_numpy()
y_24 = respondieron.loc[respondieron['ANO4'] == 2024, 'DESOCUPADOS'].to_numpy()

In [ ]:
# corremos función y exportamos a LaTeX, imprimo tablas
resultados_04 = evalua_multiples_metodos(X_04, y_04, sufijo='_04')
resultados_24 = evalua_multiples_metodos(X_24, y_24, sufijo='_24')
tabla_latex_2004 = resultados_04.to_latex(index=False)
tabla_latex_2024 = resultados_24.to_latex(index=False)

with open(OUTPUT / "tabla-2004.tex", 'w', encoding='utf-8') as f:
    f.write(tabla_latex_2004)

with open(OUTPUT / "tabla-2024.tex", 'w', encoding='utf-8') as f:
    f.write(tabla_latex_2024)

display(resultados_04)
display(resultados_24)

FIN :)